In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
import pint
import pint_xarray

In [ ]:
seapopym_v1 = xr.open_zarr(f"./zooplankton_no_transport_seapopym_v2.zarr")
seapopym_v1 = seapopym_v1["biomass"]
seapopym_v1 = seapopym_v1.sel(time=slice("2004-01-01", "2004-12-31")).load()

seapopym_v1

In [ ]:
seapopym_v2 = xr.open_zarr(f"./zooplankton_transport_full_global_seapopym_v2.zarr")
seapopym_v2 = seapopym_v2["biomass"]
seapopym_v2 = seapopym_v2.sel(time=slice("2003-01-01", "2003-12-31")).load()
seapopym_v2 = seapopym_v2.resample(time="1D").mean()
seapopym_v2

In [ ]:
# seapodym_lmtl = xr.open_zarr("../data/full_global_lmtl_forcings.zarr")
# seapodym_lmtl = seapodym_lmtl["zooplankton"]
# seapodym_lmtl = seapodym_lmtl.sel(time=slice("2004-01-01", "2004-12-31")).load()
# seapodym_lmtl


seapodym_lmtl = xr.open_zarr("./full_global_seapodym/result.zarr")
seapodym_lmtl = seapodym_lmtl["biomass"]
seapodym_lmtl = seapodym_lmtl.sel(time=slice("2004-01-01", "2004-12-31")).load()
seapodym_lmtl

In [ ]:
temperature = xr.open_zarr("../data/full_global_lmtl_forcings.zarr/")
temperature = temperature["temperature"]
temperature = temperature.sel(time=slice("2003-01-01", "2003-12-31")).load()
temperature

---

# Plot


In [ ]:
seapopym_v1 = seapopym_v1.reindex_like(seapopym_v2)
seapodym_lmtl = seapodym_lmtl.reindex_like(seapopym_v2)
temperature = temperature.reindex_like(seapopym_v2)

In [ ]:
mape_v1 = np.abs((seapopym_v1 - seapodym_lmtl) / seapodym_lmtl)
mape_v2 = np.abs((seapopym_v2 - seapodym_lmtl) / seapodym_lmtl)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

(seapodym_lmtl - seapopym_v1).mean("time").plot(vmax=1.5, ax=axes[0])
axes[0].set_title("Bias v1")

(seapodym_lmtl - seapopym_v2).mean("time").plot(vmax=1.5, ax=axes[1])
axes[1].set_title("Bias v2")

plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

((seapodym_lmtl - seapopym_v1) / seapodym_lmtl).mean("time").plot(vmax=1.5, ax=axes[0])
axes[0].set_title("MPE v1")

((seapodym_lmtl - seapopym_v2) / seapodym_lmtl).mean("time").plot(vmax=1.5, ax=axes[1])
axes[1].set_title("MPE v2")

plt.show()

In [ ]:
data = pd.DataFrame(
    {
        "mape_v1": mape_v1.resample({"time": "1MS"}).mean().data.flatten(),
        "mape_v2": mape_v2.resample({"time": "1MS"}).mean().data.flatten(),
        "temperature": temperature.resample({"time": "1MS"}).mean().data.flatten(),
    }
)
data = data[data["mape_v2"] < 4]
data = data[data["mape_v1"] < 4]
data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.histplot(
    data=data, x="temperature", y="mape_v1", cbar=True, vmax=500, ax=axes[0], cmap="viridis"
)
axes[0].set_title("MAPE between SeapoPym no transport and LMTL")

sns.histplot(
    data=data, x="temperature", y="mape_v2", cbar=True, vmax=500, ax=axes[1], cmap="viridis"
)
axes[1].set_title("MAPE between SeapoPym transport and LMTL")
plt.show()